<a href="https://colab.research.google.com/github/Nico-Araujo/FIAP/blob/main/2TIAO/Global-Solution-1/Flight_Radar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import time

def verificar_espaco_aereo_local():
    print("📡 Conectando aos radares da OpenSky Network...")

    # Coordenadas da Grande São Paulo
    lat_min, lat_max = -23.65, -23.40
    lon_min, lon_max = -46.90, -46.60

    # Endpoint da API pública
    url = f"https://opensky-network.org/api/states/all?lamin={lat_min}&lomin={lon_min}&lamax={lat_max}&lomax={lon_max}"

    try:
        resposta = requests.get(url, timeout=10)

        # O OpenSky limita requisições sem login, se der erro 429, esperamos um pouco
        if resposta.status_code == 429:
            print("⚠️ Limite da API pública atingido. Aguarde alguns segundos...")
            return True # Assume que tem voo para evitar falso positivo de UAP

        dados = resposta.json()

        voos_na_regiao = dados.get('states')

        if voos_na_regiao is None:
            print("✅ CÉU LIMPO: Nenhum voo comercial detectado nesta coordenada agora.")
            return False

        else:
            quantidade = len(voos_na_regiao)
            print(f"✈️ CÉU OCUPADO: {quantidade} aeronave(s) comercial(is) na região de SP.")

            # Mostra os detalhes básicos de quem está passando
            for voo in voos_na_regiao:
                codigo_voo = voo[1].strip() if voo[1] else "DESCONHECIDO"
                altitude = voo[7] if voo[7] else 0
                print(f"   - Voo: {codigo_voo} | Altitude: {altitude}m")

            return True # O céu está ocupado, é provável que o objeto seja um avião

    except Exception as e:
        print(f"❌ Erro de conexão com radares: {e}")
        return True # Em caso de falha de conexão, assumimos True por segurança (Fail-Safe)

# ==========================================
# TESTANDO O FILTRO
# ==========================================
if __name__ == "__main__":
    tem_aviao = verificar_espaco_aereo_local()

    print("\n--- 🧠 DECISÃO DO SISTEMA SKYGUARD ---")
    if tem_aviao:
        print("SISTEMA: O objeto detectado pela câmera provavelmente é um avião reportado. Ignorando anomalia.")
    else:
        print("SISTEMA: Objeto visual detectado, mas radares oficiais estão vazios. DISPARANDO PIPELINE DE MACHINE LEARNING!")

📡 Conectando aos radares da OpenSky Network...
✈️ CÉU OCUPADO: 2 aeronave(s) comercial(is) na região de SP.
   - Voo: PSBAS | Altitude: 1699.26m
   - Voo: PSGER | Altitude: 937.26m

--- 🧠 DECISÃO DO SISTEMA SKYGUARD ---
SISTEMA: O objeto detectado pela câmera provavelmente é um avião reportado. Ignorando anomalia.
